# 03 - Cross-System Evaluation

Now the interesting part: what happens when we test a model trained on one system on data from another?

In [ ]:
import sys
sys.path.append("../src")
import os
os.makedirs("../results/figures", exist_ok=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from preprocess import load_and_prepare
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
import warnings
warnings.filterwarnings("ignore")


In [ ]:
X_ng, y_ng, classes = load_and_prepare("../data/nigeria_students.csv")
X_pt, y_pt, _ = load_and_prepare("../data/portugal_students.csv")

rf_ng = RandomForestClassifier(n_estimators=100, random_state=42)
rf_ng.fit(X_ng, y_ng)

rf_pt = RandomForestClassifier(n_estimators=100, random_state=42)
rf_pt.fit(X_pt, y_pt)


## Transfer Test: Nigeria model on Portugal data

In [ ]:
y_pred_transfer = rf_ng.predict(X_pt)
f1_transfer_ng_to_pt = f1_score(y_pt, y_pred_transfer, average="weighted")
print(f"Nigeria -> Portugal F1: {f1_transfer_ng_to_pt:.3f}")
print(classification_report(y_pt, y_pred_transfer, target_names=classes))


## Transfer Test: Portugal model on Nigeria data

In [ ]:
y_pred_transfer2 = rf_pt.predict(X_ng)
f1_transfer_pt_to_ng = f1_score(y_ng, y_pred_transfer2, average="weighted")
print(f"Portugal -> Nigeria F1: {f1_transfer_pt_to_ng:.3f}")
print(classification_report(y_ng, y_pred_transfer2, target_names=classes))


## Feature Importance Comparison

In [ ]:
feature_names = [
    "study_hours", "attendance_rate", "parent_education",
    "has_internet", "extracurricular", "school_type",
    "teacher_student_ratio", "previous_score"
]

imp_ng = rf_ng.feature_importances_
imp_pt = rf_pt.feature_importances_

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(feature_names, imp_ng, color="#2E86AB")
axes[0].set_title("Feature Importance - Nigeria Model")
axes[0].set_xlabel("Importance")

axes[1].barh(feature_names, imp_pt, color="#A23B72")
axes[1].set_title("Feature Importance - Portugal Model")
axes[1].set_xlabel("Importance")

plt.tight_layout()
plt.savefig("../results/figures/feature_importance_comparison.png", dpi=150)
plt.show()


## Summary

The models transfer with reduced but non-trivial performance. Previous score and attendance rate remain the most stable predictors across both systems. Internet access matters much more in Nigeria than Portugal, likely because it is near-universal in Portugal and therefore does not differentiate students there.